<div style="background:linear-gradient(135deg,#1e3a8a 0%,#0c1a3d 100%);border-radius:16px;padding:30px 34px;color:#eff6ff;margin-bottom:6px;">
<div style="font-size:12.5px;letter-spacing:3px;text-transform:uppercase;opacity:.65;font-weight:600;">News Article Topic Classification</div>
<div style="font-size:30px;font-weight:800;margin:6px 0 10px;">04 · Evaluation &amp; Testing</div>
<div style="font-size:14.5px;opacity:.92;max-width:640px;line-height:1.55;">Reconstructs the Voting ensemble, predicts the evaluation set, and writes the submission file.</div>
</div>

<div style="margin:12px 0 6px;">
<span style="display:inline-block;background:#e2e8f0;color:#64748b;padding:4px 12px;border-radius:20px;font-size:11.5px;margin-right:5px;">01 Load Data</span><span style="display:inline-block;background:#e2e8f0;color:#64748b;padding:4px 12px;border-radius:20px;font-size:11.5px;margin-right:5px;">02 Preprocessing</span><span style="display:inline-block;background:#e2e8f0;color:#64748b;padding:4px 12px;border-radius:20px;font-size:11.5px;margin-right:5px;">03 Modeling</span><span style="display:inline-block;background:#1e3a8a;color:#fff;padding:4px 12px;border-radius:20px;font-size:11.5px;margin-right:5px;font-weight:600;">04 Evaluation & Testing</span><span style="display:inline-block;background:#e2e8f0;color:#64748b;padding:4px 12px;border-radius:20px;font-size:11.5px;margin-right:5px;">05 Visualization</span>
</div>

<div style="margin:8px 0 14px;">
<span style="display:inline-block;background:#eff6ff;border:1px solid #93c5fd;border-radius:8px;padding:7px 14px;font-size:13px;margin-right:8px;">
<b style="color:#1e3a8a;">⬅ Requires</b>&nbsp; <code>artifacts/02_preprocessing.pkl</code> &amp; <code>artifacts/03_modeling.pkl</code> — run those two notebooks once beforehand
</span><span style="display:inline-block;background:#eff6ff;border:1px solid #93c5fd;border-radius:8px;padding:7px 14px;font-size:13px;margin-right:8px;">
<b style="color:#1e3a8a;">➡ Produces</b>&nbsp; <code>submission.csv</code> — the deliverable
</span>
</div>

## <span style="color:#fff;">Load Artifacts</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>



In [ ]:
import numpy as np
import pandas as pd
import os
import pickle

from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Projects/DSL/News Classification/'
ARTIFACT_DIR = BASE_PATH + 'artifacts/'

with open(ARTIFACT_DIR + '03_modeling.pkl', 'rb') as f:
    _art03 = pickle.load(f)

sklearn_models       = _art03['sklearn_models']
mlp_state            = _art03['mlp_state']
mlp_arch             = _art03['mlp_arch']
weights              = _art03['weights']
evaluation_final     = _art03['evaluation_final']
evaluation_text_svd  = _art03['evaluation_text_svd']
evaluation           = _art03['evaluation']
submission           = _art03['submission']

print(f"✓ Loaded artifacts from {ARTIFACT_DIR}03_modeling.pkl — {len(sklearn_models)} classical models + MLP")
print(f"  Voting will use {len(weights)} models (weighted by validation Macro F1): {list(weights.keys())}")

## <span style="color:#fff;">Predict Evaluation Set</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Rebuilds the exact same <b>weighted</b> soft-Voting average from <code>03_modeling.ipynb</code> — only the models that cleared the validation Macro F1 floor there take part, each weighted by its own validation score (not an equal average), then argmax'd.

In [ ]:
# Re-run the exact same weighted soft-Voting average used in
# 03_modeling.ipynb -- only models that cleared the val Macro F1 floor
# there (stored in `weights`) take part; each is weighted by its own
# validation score rather than averaged equally.
proba_eval = {}
for name, model in sklearn_models.items():
    if name in weights:
        proba_eval[name] = np.asarray(model.predict_proba(evaluation_final))

if mlp_state is not None and 'MLP (TF-IDF)' in weights:
    import torch
    import torch.nn as nn

    class NewsMLP(nn.Module):
        def __init__(self, input_dim, n_classes, hidden=(256, 128)):
            super().__init__()
            layers = []
            prev = input_dim
            for h in hidden:
                layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.3)]
                prev = h
            layers.append(nn.Linear(prev, n_classes))
            self.net = nn.Sequential(*layers)

        def forward(self, x):
            return self.net(x)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    mlp = NewsMLP(mlp_arch['input_dim'], mlp_arch['n_classes'], mlp_arch['hidden']).to(device)
    mlp.load_state_dict(mlp_state)
    mlp.eval()

    with torch.no_grad():
        x_eval_t = torch.tensor(evaluation_text_svd, dtype=torch.float32).to(device)
        proba_eval['MLP (TF-IDF)'] = torch.softmax(mlp(x_eval_t), dim=1).cpu().numpy()

CLASSES_SORTED = sorted(sklearn_models[list(sklearn_models.keys())[0]].classes_)

total_w = sum(weights[name] for name in proba_eval)
avg_proba = sum(proba_eval[name] * (weights[name] / total_w) for name in proba_eval)
eval_pred = np.array(CLASSES_SORTED)[avg_proba.argmax(axis=1)]

print(f"Predictions on evaluation set : {len(eval_pred)} rows")
print(pd.Series(eval_pred).value_counts().sort_index())

## <span style="color:#fff;">Save submission.csv</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>



In [ ]:
# The evaluation set keeps its original Id (never dropped in preprocessing),
# so predictions map back to it directly.
submission_out = pd.DataFrame({
    'Id': evaluation['Id'].values,
    'Predicted': eval_pred,
})

submission_out.to_csv(BASE_PATH + 'submission.csv', index=False)
print(f"\n✓ submission.csv saved ({len(submission_out)} rows)")

submission_out.head(10)